In [41]:
!pip install tensorflow

In [42]:
import numpy as np
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, confusion_matrix
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
base = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')

x = base.output
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
out = tf.keras.layers.Dense(5, activation='softmax')(x)

classifier_model = Model(base.input, out)

for layer in base.layers:
    layer.trainable = False

C:\Users\HP\AppData\Local\Temp\ipykernel_7576\2211506567.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
).flow_from_directory(
    'dataset/train',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    'dataset/test',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 13898 images belonging to 22 classes.
Found 1546 images belonging to 22 classes.


In [ ]:
classifier_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

classifier_model.fit(train_gen, epochs=12)

Epoch 1/8
435/435 ━━━━━━━━━━━━━━━━━━━━ 861s 2s/step - accuracy: 0.1493 - loss: 2.9530
Epoch 2/8
435/435 ━━━━━━━━━━━━━━━━━━━━ 904s 2s/step - accuracy: 0.2471 - loss: 2.6119
Epoch 3/8
435/435 ━━━━━━━━━━━━━━━━━━━━ 855s 2s/step - accuracy: 0.2891 - loss: 2.4229
Epoch 4/8
435/435 ━━━━━━━━━━━━━━━━━━━━ 879s 2s/step - accuracy: 0.3137 - loss: 2.3124
Epoch 5/8
435/435 ━━━━━━━━━━━━━━━━━━━━ 849s 2s/step - accuracy: 0.3303 - loss: 2.2210
Epoch 6/8
435/435 ━━━━━━━━━━━━━━━━━━━━ 870s 2s/step - accuracy: 0.3530 - loss: 2.1494
Epoch 7/8
435/435 ━━━━━━━━━━━━━━━━━━━━ 822s 2s/step - accuracy: 0.3670 - loss: 2.1078
Epoch 8/8
435/435 ━━━━━━━━━━━━━━━━━━━━ 799s 2s/step - accuracy: 0.3744 - loss: 2.0555


In [ ]:
feature_extractor = Model(classifier.input, classifier.layers[-2].output)

In [ ]:
X_train = feature_extractor.predict(train_gen)
X_test  = feature_extractor.predict(test_gen)
y_train = train_gen.classes
y_test  = test_gen.classes

435/435 ━━━━━━━━━━━━━━━━━━━━ 889s 2s/step
49/49 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step


In [ ]:
X_train = normalize(X_train)
X_test  = normalize(X_test)

In [49]:
class GaussianBayes:
    def __init__(self):
        self.mean = {}
        self.var  = {}
        self.prior = {}

    def fit(self, X, y):
        for c in np.unique(y):
            Xc = X[y == c]
            self.mean[c] = Xc.mean(axis=0)
            self.var[c]  = Xc.var(axis=0) + 0.01
            self.prior[c] = len(Xc) / len(X)

    def predict(self, X):
        preds = []
        for x in X:
            scores = {}
            for c in self.mean:
                loglik = -0.5 * np.sum(np.log(2*np.pi*self.var[c]) + ((x - self.mean[c])**2) / self.var[c])
                scores[c] = loglik + np.log(self.prior[c])
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

In [50]:
model = GaussianBayes()
model.fit(X_train_pca, y_train)

pred = model.predict(X_test_pca)

In [51]:
print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

Accuracy: 0.37128072445019406
[[ 43   3   2   0   0   0   0   0   0   0   5   0   6   2   1   0   0   1
    0   2   0   0]
 [ 11  26   1   1   0   6   2   0   0   2   3   3  10   1   9   0   0   4
    1   0   2   1]
 [  5   0  46   0   0   0   4   4   0   0  28   1   2  10  10   0   1   6
    1   2   0   1]
 [  6   4   4   9   0   0   6   0   2   0   2   2   1   1   2   0   1   5
    0   7   2   1]
 [  4   0   2   0   7   0   2   1   2   0   0   1   0   0   1   0   5   1
    0   0   0   1]
 [  5   4   1   2   0  11   3   1   1   1   1   0   1   0   1   0   2   8
    0   4  15   0]
 [  8   0   8   0   1   0  53   1   1   0   6   4   1   0   2   0   9   9
    0   8   0   1]
 [  8   3   8   0   0   3   9   5   1   0   6   1   0   1   0   0   5   4
    0   5   1   0]
 [  4   6   4   0   2   1   6   1   4   0   5   0   1   4   8   0   4   4
    0   2   3   2]
 [  8   0   4   0   0   0   2   0   1   3   1   1   5   0   2   0   0   3
    1   1   2   0]
 [  2   1  10   0   0   0   1   1   0   

In [52]:
def top3(probs):
    return sorted(probs.items(), key=lambda x: x[1], reverse=True)[:3]

for i in range(5):
    print("Top 3 diagnosis:", top3(prob[i]))

Top 3 diagnosis: [(np.int32(0), np.float64(0.9978521324249212)), (np.int32(7), np.float64(0.0021478675750788586)), (np.int32(1), np.float64(1.562329095748894e-31))]
Top 3 diagnosis: [(np.int32(0), np.float64(1.0)), (np.int32(7), np.float64(1.2844725497043705e-20)), (np.int32(5), np.float64(1.2719831718396278e-22))]
Top 3 diagnosis: [(np.int32(0), np.float64(0.9999999999980533)), (np.int32(7), np.float64(1.9466605039307853e-12)), (np.int32(5), np.float64(2.035456880361735e-17))]
Top 3 diagnosis: [(np.int32(0), np.float64(1.0)), (np.int32(5), np.float64(6.056262698348385e-66)), (np.int32(7), np.float64(4.219791703783575e-70))]
Top 3 diagnosis: [(np.int32(0), np.float64(1.0)), (np.int32(5), np.float64(5.992158637160666e-47)), (np.int32(3), np.float64(2.793362332457054e-67))]


In [53]:
joblib.dump(model, "bayes_skin_model.pkl")
joblib.dump(train_gen.class_indices, "class_names.pkl")
feature_extractor.save("cnn_feature_model.h5")